# Domains: Custom Types with Built in Validation

Some columns always follow the same rules. An email address looks like an
email anywhere it appears, a phone number always has the same shape. Instead
of repeating the same CHECK constraint on every such column, PostgreSQL lets
you define a **domain**: a custom data type with validation built in. Define
it once, use it as a column type anywhere, and every value is validated
automatically.

## What a domain is

A domain wraps an existing type (like varchar) and adds rules. For example:

```sql
CREATE DOMAIN email_address AS varchar(255)
CHECK (VALUE ~ '^[^@]+@[^@]+\.[^@]+$');
```

Now a column of type `email_address` is really a varchar(255), but PostgreSQL
rejects any value that does not match the pattern. The keyword `VALUE` refers
to whatever is being assigned. The advantage: the rule lives in one place,
and every column using the domain inherits it. The pattern after CHECK is a
regular expression. If it looks cryptic now, do not worry, we read it piece
by piece shortly, before we build the domains.

In [ ]:
from rel_db_learning import get_project_connection, run_sql, run_ddl

project = get_project_connection()

## Growing the Library: validated contact information

Our members have no contact details yet. We will add an email and a phone
number, each validated by its own domain:

- `email_address`: a varchar(255) that must look like an email.
- `phone_number`: a varchar(20) that may contain digits, spaces, plus signs,
  hyphens, and parentheses.

As always, we design this in pgModeler, then apply it with Diff.

## Reading the validation patterns

Both domains use a **regular expression** (regex): a compact way to describe the
shape a string must have. In PostgreSQL the operator `~` means "matches this
pattern". Before building the domains, let us read the two patterns piece by
piece.

A few building blocks are enough to read both:

| Piece | Meaning |
|---|---|
| `^` | the start of the string |
| `$` | the end of the string |
| `[...]` | any one character from the set inside the brackets |
| `[^...]` | any one character NOT in the set (here `^` means "not") |
| `+` | one or more of the piece just before it |
| `\.` | a literal dot (a plain `.` means "any character", so we escape it with `\`) |

Anything not listed (like `@`) simply matches itself.

**The email pattern** `^[^@]+@[^@]+\.[^@]+$` reads, from left to right: `^` start,
`[^@]+` one or more non-`@` characters (the local part), `@` a literal at sign,
`[^@]+` one or more non-`@` characters (the domain), `\.` a dot, `[^@]+` one or
more non-`@` characters (like `com`), `$` end.

In short: something, an `@`, something, a dot, something. 

So `nikos@papadopoulos.com` passes, while by contrast
`not-an-email` has no `@` at all, so the `@` piece has nothing to match and the whole
pattern fails. That is exactly the mistake we want the domain to catch.

**The phone pattern** `^[0-9+() -]+$` is simpler: `^` start, `[0-9+() -]+` one or
more characters, each one a digit `0` to `9`, a plus sign, a parenthesis, a space,
or a hyphen, then `$` end.

In short: the whole string is made only of digits and the symbols usually seen in
phone numbers. So `+30 28220 12345` passes, while `call me` fails because it has
letters.

One habit worth keeping: the `^` at the start and the `$` at the end matter. Without
them a pattern could match just a part of the value and let extra characters slip in.
Anchoring with `^` and `$` forces the whole string to obey the rule.

This is only a first taste of regular expressions, but it is enough to read, and
even adjust, the two domains we are about to build.

### Step 1: Open and Save As

1. Open your `stage_4_design.dbm` in pgModeler.
2. Use **File > Save As** to save it as `stage_5_design.dbm` in
   `student_work/my_project/`.

### Step 2: Create the two domains

In pgModeler, a domain is a standalone object (not part of a table). Create
each one through the domain dialog (right-click the schema or use the domain
tool, then fill in the form described in the pgModeler documentation):

For **email_address**:
1. Name: `email_address`
2. Data Type: `varchar`, length `255`
3. Open the **Check constraints** tab, add a constraint:
   - Name: for example `email_format`
   - Expression: `VALUE ~ '^[^@]+@[^@]+\.[^@]+$'`

For **phone_number**:
1. Name: `phone_number`
2. Data Type: `varchar`, length `20`
3. Check constraints tab, add a constraint:
   - Name: for example `phone_format`
   - Expression: `VALUE ~ '^[0-9+() -]+$'`


### Step 3: Use the domains as column types

1. Double-click the **member** table.
2. Add a column `email`, and for its type choose the domain `email_address`
   (it appears in the type list alongside the built-in types).
3. Add a column `phone`, type `phone_number`.

### Step 4: Generate the change with Diff and apply it

1. Save `stage_5_design.dbm`.
2. Open pgModeler's Diff, source = your model, target = your my_project
   database. Generate the diff. It should contain two CREATE DOMAIN
   statements and an ALTER TABLE member adding the two columns.
3. Review the SQL, copy it, paste it below, and run it.

In [ ]:
# Paste the SQL that pgModeler's Diff generated (two CREATE DOMAIN statements
# and the ALTER TABLE member adding email and phone).
stage_5_sql = r"""
--Paste the SQL generated by pgModeler's Diff here
"""

run_ddl(stage_5_sql, project)

### Step 5: Validate Stage 5

In [ ]:
from rel_db_learning.dbm_validator import validate_exercise
from rel_db_learning.dbm_validator.exercises import PROJECT_STAGE_5_CONTACT_INFO

validate_exercise(
    project_db=get_project_connection(),
    spec=PROJECT_STAGE_5_CONTACT_INFO,
)

## The domain in action

Let's add contact details to a member. A valid email and phone are accepted:

In [ ]:
run_ddl("""
    UPDATE member
    SET email = 'nikos@papadopoulos.com', phone = '+30 28220 12345'
    WHERE member_id = 1
""", project)

run_sql("SELECT member_id, full_name, email, phone FROM member WHERE member_id = 1", project)

Now watch the domain reject an invalid email. The statement below tries to
set an email with no @ sign. It should fail with a check violation, which is
exactly the protection the domain gives us. The error is expected.

In [ ]:
run_ddl("UPDATE member SET email = 'not-an-email' WHERE member_id = 1", project)

The database refused the bad value. Without the domain, that invalid email
would have been stored. With it, the rule is enforced everywhere the domain
is used, automatically.

## Exercise

Add valid contact details to the member with member_id 2. Write an UPDATE
that sets a valid email and a valid phone for that member, then run the
check.

In [ ]:
# Write your UPDATE here, then run this cell. Use a valid email (something
# with an @ and a dot) and a valid phone (digits, spaces, +, -, parentheses).
my_update = "???"
run_ddl(my_update, project)

In [ ]:
from rel_db_learning.widgets.self_assessment import sql_query_check

sql_query_check(
    student_sql="SELECT email IS NOT NULL AND phone IS NOT NULL FROM member WHERE member_id = 2",
    correct_sql="m04_nb_12_exercise_1",
    connection=project,
)

If your email and phone were valid, the domain accepted them and the check
passes. If you used an invalid format, the UPDATE would have failed with a
check violation, and the values would still be NULL.

## What's next

Your members now have validated contact information, enforced by domains. In
the final schema stage you will learn about **transactions**, grouping
several changes so they all succeed or all fail together, and you will add
library staff and individual book copies.